Tutorial: 

`Objetivos`: 
- Convertir texto en embeddings o inscrustaciones vectoriales. 
- Añadir y eliminar documentos de la base de datos vectorial.
- Realizar búsquedas por similitud.

Fuente: 
https://www.datacamp.com/es/tutorial/chromadb-tutorial-step-by-step-guide

`ChromaDB`: es una **almacén vectorial de código abierto** que se utiliza para almacenar y recuperar **incrustaciones vectoriales(Embeddings)** de forma eficaz usando algoritmos de búsqueda por similitud (Datacamp, 2024). Estas incrustaciones representan datos(normalmente datos no estructurados como el texto) en formatos vectoriales numéricos. Chroma almacenará el texto y se encargará de la incrustación y la indexación automáticamente. Las **colecciones** son el lgugar en el chroma permiten almacenar las incrustaciones, los documentos y los metadatos. De forma predeterminada, Chroma convierte el texto a incrustaciones con el modelo de embedding `all-MiniLM-L6-v2`. 

`**Cosine Similarity o Similitud de Coseno**` es una métrica de similutud que determina si dos vectores son similar entre sí.
La similitud del coseno es el coseno del ángulo entre vectores. El rango de similitud está entre -1 y 1, donde -1 son vectores 
absolutamente opuestos, 0 no hay correlación, 1 son absolutamente similares.

![Cosine Similarity](https://miro.medium.com/v2/resize:fit:828/format:webp/1*dyH20eCqb6qTL-gt4nCVzQ.png)

`**Cosine Distance o Distancias del Coseno**` :  $distancia del coseno = 1 — similitud del coseno$

Fuente: https://www.datacamp.com/tutorial/cosine-distance

La distancia coseno mide la disimilitud entre dos vectores , es decir, cuanto de diferentes son los dos vectores.
El rango de la distancia del coseno está entre 0 y 2, donde 0  significa que los vectores estan perfectamente alineados (**máxima similitud**) , 2 
significa que son opuestos(**máxima disimilitud**)

### Instalar 

`PIP`
```bash
pip install chromadb
pip install chroma-hnswlib
```
`POETRY`
```bash
poetry add chromadb 
poetry add chroma-hnswlib
```


## `Caso1`: Base de datos vectorial guardada temporalmente en memoria.
Al cerrar el notebook la base se datos no se guadada dado que la memoria RAM es volátil.

### Crea un cliente Chroma

In [1]:
import chromadb

# Create a client 
chroma_client = chromadb.Client()

### Crear una collección de documentos.

In [2]:
# Create a empty collection 
collection = chroma_client.create_collection(name="my_collection", metadata={"hnsw:space": "cosine"})

In [3]:
print(f'Name of collection: {collection.name}')
print(f'Total number of embeddings: {collection.count()}')

Name of collection: my_collection
Total number of embeddings: 0


### Agregar documentos a la colección

In [4]:
# Add some text documents to the collection

collection.add(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1", "id2"]
)

### Consulta la colección

In [5]:
### Query the collection
results = collection.query(
    query_texts=["This is a query document about hawaii"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
print(results)

{'ids': [['id1', 'id2']], 'embeddings': None, 'documents': [['This is a document about pineapple', 'This is a document about oranges']], 'uris': None, 'data': None, 'metadatas': [[None, None]], 'distances': [[0.5202003717422485, 0.6215400695800781]], 'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


In [6]:
### Query the collection
results = collection.query(
    query_texts=["This is a query document about hawaii"], # Chroma will embed this for you
    n_results=1 # how many results to return
)
print(results)

{'ids': [['id1']], 'embeddings': None, 'documents': [['This is a document about pineapple']], 'uris': None, 'data': None, 'metadatas': [[None]], 'distances': [[0.5202003717422485]], 'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


## `Caso2`: Base de datos vectorial guardada en el disco local.
Al cerrar el notebook la base se datos persiste en el tiempo.

In [7]:

client = chromadb.PersistentClient(path="../vectorstore")
# Get a collection object from an existing collection, by name. If it doesn't exist, create it.
collection = client.get_or_create_collection(name="my_collection", metadata={"hnsw:space": "cosine"})

# Add some text documents to the collection
collection.add(
    documents=[
        "A chatbot is a computer program that simulates and processes human conversation (either written or spoken), allowing humans to interact with digital devices as if they were communicating with a real person.",
        "A vector database, vector store or vector search engine is a database that can store vectors (fixed-length lists of numbers) along with other data items.",
        "A vector store is a specialized system designed for holding embedded vectors.",
        "Embedding is a means of representing objects like text, images and audio as points in a continuous vector space where the locations of those points in space are semantically meaningful to machine learning (ML) algorithms",
        "A large language model (LLM) definition is a type of machine learning (ML) model that can perform a variety of natural language processing (NLP) task"
    ],
    ids=["id1", "id2", "id3", "id4", "id5"]
)


In [8]:
### Query the collection
results = collection.query(
    query_texts=["what is a vector database?"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
print(results)

{'ids': [['id2', 'id3']], 'embeddings': None, 'documents': [['A vector database, vector store or vector search engine is a database that can store vectors (fixed-length lists of numbers) along with other data items.', 'A vector store is a specialized system designed for holding embedded vectors.']], 'uris': None, 'data': None, 'metadatas': [[None, None]], 'distances': [[0.14280283157232587, 0.3306864680642785]], 'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


In [11]:
print(results['ids'])
print(results['distances'])
print(f'Total number of embeddings: {collection.count()}')
print(f"Documento relevante 1:{results['documents'][0][0]}")
print(f"Documento relevante 2:{results['documents'][0][1]}")

[['id2', 'id3']]
[[0.14280283157232587, 0.3306864680642785]]
Total number of embeddings: 5
Documento relevante 1:A vector database, vector store or vector search engine is a database that can store vectors (fixed-length lists of numbers) along with other data items.
Documento relevante 2:A vector store is a specialized system designed for holding embedded vectors.
